# Gamma Pattern Validation of Spectral Absorption Rate Distributions

## 0. Input overview

The pipeline needs **one measured** and **one reference** SAR dataset in CSV format.

### Folder structure (example)

```
data/example/
├── measured_data/
│   └── measured_SAR.csv
└── reference_data/
    └── reference_SAR.csv
```

### CSV format

Each CSV must contain at least:

- **Coordinates:** `x`, `y`  
  Any header beginning with `x` or `y` is accepted. Examples:
  - `x`
  - `x_mm`
  - `x [mm]`
  - `y`
  - `y_mm`
  - `y [mm]`

- **SAR column:** either `sar` **or** any column whose name contains `"sar"`  
  (e.g. `pssar1g`, `pssar10g`, `sSAR_1g`, etc.)

The loader auto-detects the SAR column.

### Units (important)

- Coordinate units must be consistent between measured and reference.
- The loader **does not** rescale to match datasets automatically.
- `resample_resolution` is interpreted in the same physical units as `x,y`.

In [ ]:
# --- Paths: measured + reference CSVs ---

measured_csv = "data/example/measured_sSAR1g.csv"
reference_csv = "data/example/reference_sSAR1g.csv"

measured_csv, reference_csv

## 1. Loader overview

---

The loader builds two versions of each dataset:

- **linear** images (peak-normalized, used for gamma)
- **dB** images (log-compressed, used for registration)

### Conventions

- **reference** = fixed image (target grid / orientation preserved)
- **measured**  = moving image (warped onto the reference grid)

---

### Masking and normalization (current implementation)

1) Absolute cutoff in W/kg:

```
cutoff_abs_wkg = min(0.1, 2 * noise_floor_wkg)
```

2) Absolute masks:

```
mask_abs = (sar_wkg >= cutoff_abs_wkg)
```

3) Peak normalization (by masked peak, separately for measured/reference):

- Linear images keep all values **inside support** (no hard cutoff), normalized by masked peak  
- Values **outside support** are set to 0

### dB conversion (for registration)

- dB images are derived from the linear images.
- Before `log10`, values are clamped to a floor derived from the absolute cutoff:

```
floor_norm = cutoff_abs_wkg / peak_abs_wkg
```

- A shared floor is applied so both images have a compatible range for MI.

```
shared_floor_norm = max(measured_floor_norm, reference_floor_norm, 1e-12)
```

---

### Key loader settings

- `noise_floor_wkg`  
  Absolute noise floor in W/kg used to compute the cutoff:

  ```
  cutoff_abs_wkg = min(0.1, 2 * noise_floor_wkg)
  ```

- `resample_resolution`  
  If set, resamples each CSV onto a uniform grid with linear interpolation.  
  If `None`, uses the native lattice via pivot-table gridding; missing points are filled with zeros.



In [ ]:
# --- Load SAR data as images ---
from sar_pattern_validation import SARImageLoader

loader = SARImageLoader(
    measured_path=str(measured_csv),
    reference_path=str(reference_csv),
    show_plot=True,
    resample_resolution=None,
    noise_floor_wkg=0.5,
)

# Loader convention (THIS loader):
#   fixed  = measured
#   moving = reference
reference_image_db, measured_image_db = loader.get_images()

# Linear-space images (for masks/metrics + gamma)
measured_image_linear = loader.measured_image_linear  # fixed
reference_image_linear = loader.reference_image_linear  # moving

## 2. Registration overview

---

Registration runs on **dB images**, but uses the same geometry as the linear images.

### Conventions (current implementation)

- **fixed**   = **reference** (target grid)
- **moving**  = **measured** (warped onto fixed)
- **aligned** = **measured resampled onto the reference grid**

---

### Pipeline steps

1. Cast images to `Float32`.
2. Initialize the transform using **center-of-gravity (Moments)**.
3. Metric: **Mattes Mutual Information** (MI).
4. Stage-defined multi-scale pyramid (default workflow profile):
   - coarse: shrink `[8, 4, 2]`, smoothing `[3.0, 1.5, 0.0]`
   - medium: shrink `[4, 2, 1]`, smoothing `[2.0, 1.0, 0.0]`
   - fine: shrink `[2, 1]`, smoothing `[1.0, 0.0]`
5. Default strategy: **Hybrid**
   - coarse global exploration from stage settings
   - local gradient-descent line-search refinement per stage
6. Execute: returns the **moving** image resampled on the **fixed** grid, plus the final transform.

### Mask policy (current implementation)

Registration attempts in order (per stage):

1. fixed + moving masks (if both provided)
2. fixed mask only (recommended default)
3. no masks

If MI fails with:

```
All samples map outside moving image buffer
```

it falls back to the next less restrictive masking option.

---

## Key parameters (registration)

- `transform_type`: `"translate" | "rigid" | "affine"` (default: `"rigid"`)
- `rot_step_deg`: rotation sampling step (degrees)
- `rot_span_deg`: total rotation span (degrees, centered on init)
- `tx_steps`, `ty_steps`: translation step counts per axis (symmetric)
- `translation_step`: translation step size in physical units (same units as image spacing)

### Practical sizing

- Set `translation_step` close to pixel size (e.g. `0.001` for a 1 mm grid).
- Choose `tx_steps`, `ty_steps` so `tx_steps * translation_step` covers expected residual translation after COG init.
- Default profile uses three stages:
  - coarse: wider span to capture gross mismatch
  - medium: narrower span with stronger local refinement
  - fine: smallest span for final convergence

In [ ]:
from sar_pattern_validation import Rigid2DRegistration, show_registration_overlay
from sar_pattern_validation.workflow_config import WorkflowConfig

# --- Masks (linear) ---
measured_mask, reference_mask = loader.make_metric_masks()

# --- Registration defaults from workflow config (optimized hybrid profile) ---
registration_defaults = WorkflowConfig()
stages = registration_defaults.stages

# --- Registration in dB space ---
# fixed_image  = measured (kept in measurement coordinates)
# moving_image = reference (warped onto measured grid)
registration = Rigid2DRegistration(
    fixed_image=measured_image_db,
    moving_image=reference_image_db,
    transform_type=registration_defaults.transform_type,
)

# Transform maps reference → measured
aligned_reference_db, reference_to_measured_tx = registration.run(
    stages=stages,
    # fixed_mask=measured_mask,
    moving_mask=reference_mask,
)

# --- Overlay in dB ---
show_registration_overlay(
    measured_image_db,  # red   = measured
    aligned_reference_db,  # blue  = reference → measured
    title="Overlay in dB (measured vs reference→measured)",
)

## 3. GammaMapEvaluator overview

The `GammaMapEvaluator` computes a **2D gamma index** between a **measured SAR map** and a **reference SAR map** after registration.

### Conventions (current implementation)

- **reference_sar_linear**: reference (fixed grid)
- **measured_sar_linear**: measured (native grid)
- `reference_to_measured_transform`: reference → measured transform

Gamma is computed **on the measured grid** after resampling the reference map onto the measured grid.

All computations are performed in **linear SAR** (peak-normalized inputs).

---

## Gamma pipeline

1) **Resample measured onto reference**

- reference is resampled onto the measured grid using `reference_to_measured_transform`
- linear interpolation
- outside-domain default is 0.0

2) **Convert tolerances**

- Distance-to-agreement in mm is converted to an integer pixel radius:

```
dta_pixels = round((distance_mm / 1000) / min_spacing_m)
```

- Dose-to-agreement in percent is converted to a fraction of peak:

```
dose_tol_fraction = dose_to_agreement_percent / 100
```

3) **Evaluation region (ROI / masking)**

If masks are provided:

```
evaluation_mask = reference_mask ∩ resampled(measured_mask)
```

If neither mask is provided, gamma is evaluated everywhere.

4) **Outputs**

- `gamma_map`: float array with NaN outside `evaluation_mask` (if masks used)
- `pass_rate_percent`: percent of evaluated pixels with γ ≤ 1
- `evaluated_pixel_count`, `passed_pixel_count`, `failed_pixel_count`

---

## Gamma equation (as implemented)

For each evaluated pixel (on the measured grid), gamma is:

$$
\gamma(\mathbf{r}) = \min_{\mathbf{r}' \in \mathcal{N}(\mathbf{r})}
\sqrt{
\left(\frac{\|\mathbf{r}-\mathbf{r}'\|}{\mathrm{DTA}}\right)^2 +
\left(\frac{ SAR_\text{eval}(\mathbf{r}) - SAR_\text{ref}(\mathbf{r}') }{ \Delta SAR_\mathrm{tol} }\right)^2
}
$$

Where:

- DTA is applied in pixel units (`dta_pixels`)
- Delta SAR is the **global fraction of peak**, because inputs are peak-normalized:

$$
\Delta SAR_\mathrm{tol} = \frac{\text{dose\_to\_agreement\_percent}}{100}
$$

- The search neighborhood is a disk with radius `dta_pixels`.

---

## Key parameters (GammaMapEvaluator)

- `dose_to_agreement_percent`  
  Global tolerance as a percent of peak (peak-normalized inputs).

- `distance_to_agreement_mm`  
  DTA in mm, internally converted to an integer pixel radius using reference spacing.

- `gamma_cap`  
  Cap used for output/visualization (values above are clipped).

- `reference_mask_u8`, `measured_mask_u8` (optional)  
  Used only to define the evaluation region:
  - reference mask is used directly
  - measured mask is resampled to the reference grid and intersected

In [ ]:
from sar_pattern_validation import GammaMapEvaluator

reference_image_linear = loader.reference_image_linear
measured_image_linear = loader.measured_image_linear

gamma_eval = GammaMapEvaluator(
    reference_sar_linear=reference_image_linear,
    measured_sar_linear=measured_image_linear,
    reference_to_measured_transform=reference_to_measured_tx,
    dose_to_agreement_percent=5.0,  # ΔD (% of peak) — paper default example
    distance_to_agreement_mm=2.0,  # Δd (mm)       — paper recommendation
    gamma_cap=2.0,
)

# Optional ROI (recommended): intersection of reference mask and resampled measured mask
gamma_eval.reference_mask_u8 = reference_mask
gamma_eval.measured_mask_u8 = measured_mask

gamma_eval.compute()
gamma_eval.show()